# Notebook 02 — Staging & Qualité des données
## Projet MSPR · Electio-Analytics · Pays de la Loire

---

### 🎯 Objectif de ce notebook

Ce notebook correspond à la **deuxième étape du pipeline ETL** : le **staging et le contrôle qualité**.

Il prend en entrée les fichiers **bruts** (`stg_raw_*.csv`) produits par le notebook 01,
et produit deux types de sorties :

- `stg_std_*.csv` → données **standardisées et nettoyées**, prêtes pour les transformations
- `stg_reject_*.csv` → lignes **rejetées** avec le motif de rejet (jamais supprimées en silence !)

### 📋 Les 4 familles de règles qualité appliquées

Conformément au cours ETL, on contrôle 4 dimensions :

| Famille | Question | Exemple |
|---|---|---|
| **Complétude** | Le champ obligatoire est-il présent ? | `code_departement` non null |
| **Cohérence** | La valeur a-t-elle un sens métier ? | `inscrits > 0`, `votants <= inscrits` |
| **Validité** | Le format / référentiel est-il respecté ? | `code_departement` dans la liste PDL |
| **Unicité** | Y a-t-il des doublons interdits ? | Même bureau de vote × même élection |

### 🔄 Cycle de traitement par dataset
```
stg_raw  →  [Typage]  →  [Standardisation]  →  [Contrôles qualité]  →  stg_std
                                                        ↓
                                                   stg_reject
```

---
### 👤 Réalisé par : Mickeal (Data Engineer)
### 📅 Étape : 2/4 — Staging → `stg_std` + `stg_reject`


---
## 0. Imports et configuration

In [8]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Imports OK")


✅ Imports OK


In [9]:
# ── Constantes du projet ──────────────────────────────────────────────────
ZONE_ETUDE = "Pays de la Loire"
DEPTS_PDL  = ['44', '49', '53', '72', '85']

ELECTIONS_CIBLES = [
    '2012_pres_t1', '2017_pres_t1', '2022_pres_t1',
    '2012_legi_t1', '2017_legi_t1', '2022_legi_t1',
]

ROOT         = ".."
PATH_STG_RAW    = os.path.join(ROOT, "outputs", "staging", "raw")
PATH_STG_STD    = os.path.join(ROOT, "outputs", "staging", "std")
PATH_STG_REJECT = os.path.join(ROOT, "outputs", "staging", "reject")
PATH_STAGING    = os.path.join(ROOT, "outputs", "staging")
PATH_OPS     = os.path.join(ROOT, "outputs", "ops")

os.makedirs(PATH_STG_RAW,    exist_ok=True)
os.makedirs(PATH_STG_STD,    exist_ok=True)
os.makedirs(PATH_STG_REJECT, exist_ok=True)
os.makedirs(PATH_OPS,        exist_ok=True)

# Seuils d'alerte qualité
SEUIL_REJET_MAX  = 0.10   # alerte si taux de rejet > 10%
SEUIL_NULL_MAX   = 0.05   # alerte si taux de nulls > 5% par colonne

BATCH_ID    = f"B02_STAGING_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
BATCH_START = datetime.now()

print(f"🚀 Batch démarré : {BATCH_ID}")


🚀 Batch démarré : B02_STAGING_20260523_213217


---
## 1. Fonctions utilitaires — Qualité & Rejets

In [10]:
# ── Fonction : score de qualité d'un DataFrame ───────────────────────────
def score_qualite(df, nom):
    """
    Calcule et affiche le score de qualité d'un DataFrame.
    
    Mesure 3 indicateurs :
    - Complétude  : % de cellules non nulles
    - Unicité     : % de lignes non dupliquées
    - Score global: moyenne pondérée des deux
    
    Retourne le score global (entre 0 et 1).
    """
    nb_cellules   = df.shape[0] * df.shape[1]
    nb_nulls      = int(df.isnull().sum().sum())
    completude    = 1 - (nb_nulls / nb_cellules)
    
    nb_doublons   = df.duplicated().sum()
    unicite       = 1 - (nb_doublons / len(df)) if len(df) > 0 else 1
    
    score_global  = (completude * 0.6) + (unicite * 0.4)
    
    print(f"  📊 Score qualité — {nom}")
    print(f"     Complétude  : {completude*100:.2f}% ({nb_nulls:,} nulls sur {nb_cellules:,} cellules)")
    print(f"     Unicité     : {unicite*100:.2f}% ({nb_doublons:,} doublons)")
    print(f"     Score global: {score_global*100:.2f}%")
    
    # Alerte si score faible
    if score_global < 0.90:
        print(f"  ⚠️  ALERTE : score qualité < 90% !")
    else:
        print(f"  ✅ Qualité acceptable")
    print()
    
    return round(score_global, 4)


# ── Fonction : créer un enregistrement de rejet ───────────────────────────
def creer_rejet(df_rejets, batch_id, source, rule_code, rule_reason, df_lignes):
    """
    Ajoute les lignes rejetées dans le DataFrame de rejets.
    
    Conformément au cours ETL, une ligne rejetée est JAMAIS supprimée en silence.
    Elle est isolée avec : batch_id, source, règle en échec, payload brut, timestamp.
    
    Paramètres :
        df_rejets  : DataFrame existant des rejets (à compléter)
        batch_id   : identifiant du batch en cours
        source     : nom du fichier source
        rule_code  : code de la règle qualité (ex: 'R01')
        rule_reason: description de la règle
        df_lignes  : DataFrame des lignes à rejeter
    
    Retourne le DataFrame de rejets mis à jour.
    """
    if len(df_lignes) == 0:
        return df_rejets
    
    nouveaux_rejets = pd.DataFrame({
        'batch_id'      : batch_id,
        'source_fichier': source,
        'reject_rule'   : rule_code,
        'reject_reason' : rule_reason,
        'nb_lignes'     : len(df_lignes),
        'raw_payload'   : df_lignes.astype(str).apply(
                            lambda r: r.to_json(), axis=1
                          ).values,
        'reject_ts'     : datetime.now().isoformat()
    })
    
    return pd.concat([df_rejets, nouveaux_rejets], ignore_index=True)


# ── Fonction : rapport de standardisation ────────────────────────────────
def rapport_std(df_raw, df_std, df_rejet, nom):
    """
    Affiche un rapport de la standardisation :
    - Nb lignes en entrée
    - Nb lignes valides (stg_std)
    - Nb lignes rejetées (stg_reject)
    - Taux de rejet
    """
    n_entree  = len(df_raw)
    n_valides = len(df_std)
    n_rejets  = len(df_rejet[df_rejet['source_fichier'] == nom]) if len(df_rejet) > 0 else 0
    taux      = n_rejets / n_entree * 100 if n_entree > 0 else 0
    
    print(f"  📋 Rapport staging — {nom}")
    print(f"     Entrée (stg_raw)  : {n_entree:,} lignes")
    print(f"     Valides (stg_std) : {n_valides:,} lignes")
    print(f"     Rejetées          : {n_rejets:,} lignes")
    print(f"     Taux de rejet     : {taux:.2f}%")
    
    if taux > SEUIL_REJET_MAX * 100:
        print(f"  ⚠️  ALERTE : taux de rejet > {SEUIL_REJET_MAX*100:.0f}% !")
    else:
        print(f"  ✅ Taux de rejet acceptable")
    print()


print("✅ Fonctions qualité définies")


✅ Fonctions qualité définies


---
## 2. Staging — `stg_raw_general.csv`
### Participation électorale

**Colonnes clés à contrôler :**
- `id_election` : doit être dans la liste des élections cibles
- `code_departement` : doit être dans les 5 depts PDL
- `inscrits` : doit être > 0 (sinon le bureau de vote est vide)
- `votants` : doit être >= 0 ET <= inscrits
- `ratio_votants_inscrits` : doit être entre 0 et 100

**Règles de rejet :**
- R01 : `code_departement` null ou hors PDL
- R02 : `id_election` null ou hors liste cibles
- R03 : `inscrits` null ou <= 0
- R04 : `votants` > `inscrits` (incohérence métier)
- R05 : doublon sur (`id_election` + `code_departement` + `code_bv`)


In [11]:
# ── Chargement stg_raw_general ───────────────────────────────────────────
df_gen_raw = pd.read_csv(
    os.path.join(PATH_STG_RAW, "stg_raw_general.csv"),
    dtype={'code_departement': str},
    sep=','
)

print(f"📂 Chargé : stg_raw_general.csv — {len(df_gen_raw):,} lignes")
print()
score_qualite(df_gen_raw, "stg_raw_general")


📂 Chargé : stg_raw_general.csv — 19,824 lignes

  📊 Score qualité — stg_raw_general
     Complétude  : 88.11% (58,941 nulls sur 495,600 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 92.86%
  ✅ Qualité acceptable



np.float64(0.9286)

In [12]:
# ── Typage et standardisation ────────────────────────────────────────────
df_gen_std   = df_gen_raw.copy()
df_gen_rejet = pd.DataFrame()   # DataFrame vide pour accumuler les rejets

# 1. Normaliser code_departement : strip + zfill(2)
#    Ex: " 44 " → "44", "4" → "04"
df_gen_std['code_departement'] = (
    df_gen_std['code_departement']
    .astype(str).str.strip().str.zfill(2)
)

# 2. Standardiser id_election : strip + lower
#    Ex: " 2022_PRES_T1 " → "2022_pres_t1"
df_gen_std['id_election'] = (
    df_gen_std['id_election']
    .astype(str).str.strip().str.lower()
)

# 3. Typer les colonnes numériques
cols_numeriques = ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes',
                   'ratio_abstentions_inscrits', 'ratio_votants_inscrits']

for col in cols_numeriques:
    if col in df_gen_std.columns:
        df_gen_std[col] = pd.to_numeric(df_gen_std[col], errors='coerce')

print("✅ Typage et standardisation appliqués")
print(f"   Types après standardisation :")
print(df_gen_std[['id_election','code_departement','inscrits','votants',
                   'ratio_votants_inscrits']].dtypes.to_string())


✅ Typage et standardisation appliqués
   Types après standardisation :
id_election                   str
code_departement              str
inscrits                    int64
votants                     int64
ratio_votants_inscrits    float64


In [13]:
# ── Reconstruction des colonnes manquantes par formule ────────────────────
# Avant d applique les règles qualité, on tente de RECONSTRUIRE
# les valeurs manquantes grâce à la relation mathématique :
#
#   votants = exprimes + blancs + nuls
#
# Si une seule des 3 colonnes est NaN mais que les 2 autres sont connues,
# on peut la recalculer exactement plutôt que de rejeter la ligne.
#
# Exemple :
#   votants=800, exprimes=750, nuls=10, blancs=NaN
#   → blancs = 800 - 750 - 10 = 40  ✅ ligne sauvée !

nb_reconstruit = 0

# ── Cas 1 : blancs est NaN mais votants, exprimes et nuls sont connus ─────
masque_blancs_nan = (
    df_gen_std["blancs"].isnull() &
    df_gen_std["votants"].notnull() &
    df_gen_std["exprimes"].notnull() &
    df_gen_std["nuls"].notnull()
)
if masque_blancs_nan.any():
    df_gen_std.loc[masque_blancs_nan, "blancs"] = (
        df_gen_std.loc[masque_blancs_nan, "votants"]   -
        df_gen_std.loc[masque_blancs_nan, "exprimes"]  -
        df_gen_std.loc[masque_blancs_nan, "nuls"]
    )
    nb_reconstruit += masque_blancs_nan.sum()
    print(f"  ✅ blancs reconstruit   : {masque_blancs_nan.sum():,} lignes")

# ── Cas 2 : nuls est NaN mais votants, exprimes et blancs sont connus ─────
masque_nuls_nan = (
    df_gen_std["nuls"].isnull() &
    df_gen_std["votants"].notnull() &
    df_gen_std["exprimes"].notnull() &
    df_gen_std["blancs"].notnull()
)
if masque_nuls_nan.any():
    df_gen_std.loc[masque_nuls_nan, "nuls"] = (
        df_gen_std.loc[masque_nuls_nan, "votants"]   -
        df_gen_std.loc[masque_nuls_nan, "exprimes"]  -
        df_gen_std.loc[masque_nuls_nan, "blancs"]
    )
    nb_reconstruit += masque_nuls_nan.sum()
    print(f"  ✅ nuls reconstruit     : {masque_nuls_nan.sum():,} lignes")

# ── Cas 3 : exprimes est NaN mais votants, blancs et nuls sont connus ─────
masque_expr_nan = (
    df_gen_std["exprimes"].isnull() &
    df_gen_std["votants"].notnull() &
    df_gen_std["blancs"].notnull() &
    df_gen_std["nuls"].notnull()
)
if masque_expr_nan.any():
    df_gen_std.loc[masque_expr_nan, "exprimes"] = (
        df_gen_std.loc[masque_expr_nan, "votants"]  -
        df_gen_std.loc[masque_expr_nan, "blancs"]   -
        df_gen_std.loc[masque_expr_nan, "nuls"]
    )
    nb_reconstruit += masque_expr_nan.sum()
    print(f"  ✅ exprimes reconstruit : {masque_expr_nan.sum():,} lignes")

# ── Résultat ───────────────────────────────────────────────────────────────
if nb_reconstruit > 0:
    print(f"Total lignes sauvées par reconstruction : {nb_reconstruit:,}")
    print(f"  Ces lignes auraient été rejetées en R04bis sans cette étape.")
else:
    print(f"  Aucune reconstruction nécessaire — colonnes déjà complètes ✅")


  ✅ blancs reconstruit   : 6,430 lignes
Total lignes sauvées par reconstruction : 6,430
  Ces lignes auraient été rejetées en R04bis sans cette étape.


In [14]:
# ── Contrôles qualité et extraction des rejets ───────────────────────────
source_nom = "stg_raw_general.csv"

# ── R01 : code_departement null ou hors PDL ───────────────────────────────
masque_r01 = (
    df_gen_std["code_departement"].isnull() |
    ~df_gen_std["code_departement"].isin(DEPTS_PDL)
)
if masque_r01.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R01", "code_departement null ou hors Pays de la Loire",
        df_gen_std[masque_r01]
    )
    df_gen_std = df_gen_std[~masque_r01].copy()
    print(f"  R01 : {masque_r01.sum():,} lignes rejetées (dept hors PDL)")

# ── R02 : id_election null ou hors liste cibles ───────────────────────────
masque_r02 = (
    df_gen_std["id_election"].isnull() |
    ~df_gen_std["id_election"].isin(ELECTIONS_CIBLES)
)
if masque_r02.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R02", "id_election null ou hors élections cibles",
        df_gen_std[masque_r02]
    )
    df_gen_std = df_gen_std[~masque_r02].copy()
    print(f"  R02 : {masque_r02.sum():,} lignes rejetées (élection hors cibles)")

# ── R03 : inscrits null ou <= 0 ───────────────────────────────────────────
masque_r03 = (
    df_gen_std["inscrits"].isnull() |
    (df_gen_std["inscrits"] <= 0)
)
if masque_r03.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R03", "inscrits null ou <= 0 (bureau vide)",
        df_gen_std[masque_r03]
    )
    df_gen_std = df_gen_std[~masque_r03].copy()
    print(f"  R03 : {masque_r03.sum():,} lignes rejetées (inscrits <= 0)")

# ── R04 : votants > inscrits (incohérence métier) ─────────────────────────
masque_r04 = df_gen_std["votants"] > df_gen_std["inscrits"]
if masque_r04.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R04", "votants > inscrits — incohérence métier impossible",
        df_gen_std[masque_r04]
    )
    df_gen_std = df_gen_std[~masque_r04].copy()
    print(f"  R04 : {masque_r04.sum():,} lignes rejetées (votants > inscrits)")

# ── R04bis : exprimes + blancs + nuls != votants ──────────────────────────
# Cette règle vérifie la cohérence interne des colonnes électorales.
# Par définition : votants = exprimes + blancs + nuls
# Si cette équation n est pas respectée, c est une erreur de saisie
# ou une corruption du fichier source qu on doit isoler.
#
# On utilise une tolérance de +-1 pour les arrondis éventuels
masque_r04bis = (
    abs(
        df_gen_std["exprimes"] +
        df_gen_std["blancs"]   +
        df_gen_std["nuls"]     -
        df_gen_std["votants"]
    ) > 1   # tolérance de 1 pour les arrondis
)
if masque_r04bis.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R04bis",
        "exprimes + blancs + nuls != votants — incohérence interne bureau de vote",
        df_gen_std[masque_r04bis]
    )
    df_gen_std = df_gen_std[~masque_r04bis].copy()
    print(f"  R04bis : {masque_r04bis.sum():,} lignes rejetées (exprimes+blancs+nuls != votants)")
else:
    print(f"  R04bis : 0 rejet — cohérence exprimes+blancs+nuls=votants vérifiée ✅")

# ── R05 : doublons sur clé métier ─────────────────────────────────────────
cle_metier    = ["id_election", "code_departement", "code_bv"]
cle_existante = [c for c in cle_metier if c in df_gen_std.columns]
masque_r05    = df_gen_std.duplicated(subset=cle_existante, keep="first")
if masque_r05.any():
    df_gen_rejet = creer_rejet(
        df_gen_rejet, BATCH_ID, source_nom,
        "R05", f"Doublon sur clé métier {cle_existante}",
        df_gen_std[masque_r05]
    )
    df_gen_std = df_gen_std[~masque_r05].copy()
    print(f"  R05 : {masque_r05.sum():,} lignes rejetées (doublons)")

print()
rapport_std(df_gen_raw, df_gen_std, df_gen_rejet, source_nom)


  R04bis : 0 rejet — cohérence exprimes+blancs+nuls=votants vérifiée ✅
  R05 : 16,596 lignes rejetées (doublons)

  📋 Rapport staging — stg_raw_general.csv
     Entrée (stg_raw)  : 19,824 lignes
     Valides (stg_std) : 3,228 lignes
     Rejetées          : 16,596 lignes
     Taux de rejet     : 83.72%
  ⚠️  ALERTE : taux de rejet > 10% !



In [15]:
# ── Ajout de colonnes calculées utiles ───────────────────────────────────
# On calcule le taux de participation proprement si la colonne n'est pas fiable
df_gen_std['taux_participation'] = (
    df_gen_std['votants'] / df_gen_std['inscrits'] * 100
).round(4)

df_gen_std['taux_abstention'] = (
    df_gen_std['abstentions'] / df_gen_std['inscrits'] * 100
).round(4)

# Extraction de l'année et du type d'élection depuis id_election
# Ex: "2022_pres_t1" → annee=2022, type_election="pres"
df_gen_std['annee']          = df_gen_std['id_election'].str[:4].astype(int)
df_gen_std['type_election']  = df_gen_std['id_election'].str[5:9]  # "pres" ou "legi"

print("✅ Colonnes calculées ajoutées :")
print("   taux_participation, taux_abstention, annee, type_election")
print()
print("  Aperçu final stg_std_general :")
print(df_gen_std[['id_election','annee','type_election','code_departement',
                   'inscrits','votants','taux_participation']].head(5).to_string(index=False))


✅ Colonnes calculées ajoutées :
   taux_participation, taux_abstention, annee, type_election

  Aperçu final stg_std_general :
 id_election  annee type_election code_departement  inscrits  votants  taux_participation
2022_pres_t1   2022          pres               44      1553     1177             75.7888
2022_pres_t1   2022          pres               44      1046      818             78.2027
2022_pres_t1   2022          pres               44      1011      795             78.6350
2022_pres_t1   2022          pres               44       941      727             77.2582
2022_pres_t1   2022          pres               44       890      695             78.0899


In [16]:
# ── Sauvegarde stg_std_general.csv ───────────────────────────────────────
df_gen_std.to_csv(
    os.path.join(PATH_STG_STD, "stg_std_general.csv"),
    index=False, encoding='utf-8'
)

# Sauvegarde stg_reject_general.csv (même si vide, on crée le fichier)
df_gen_rejet.to_csv(
    os.path.join(PATH_STG_REJECT, "stg_reject_general.csv"),
    index=False, encoding='utf-8'
)

print(f"💾 stg_std_general.csv    → {len(df_gen_std):,} lignes")
print(f"💾 stg_reject_general.csv → {len(df_gen_rejet):,} lignes rejetées")


💾 stg_std_general.csv    → 3,228 lignes
💾 stg_reject_general.csv → 16,596 lignes rejetées


---
## 3. Staging — `stg_raw_candidats.csv`
### Résultats par candidat

**Règles de rejet :**
- R01 : `code_departement` null ou hors PDL
- R02 : `id_election` null ou hors liste cibles
- R06 : `nuance` null (on ne peut pas identifier la tendance politique)
- R07 : `voix` null ou < 0
- R08 : `ratio_voix_exprimes` hors plage [0, 100]


In [17]:
# ── Chargement et standardisation candidats ──────────────────────────────
df_cand_raw = pd.read_csv(
    os.path.join(PATH_STG_RAW, "stg_raw_candidats.csv"),
    dtype={'code_departement': str},
    sep=','
)

print(f"📂 Chargé : stg_raw_candidats.csv — {len(df_cand_raw):,} lignes")
score_qualite(df_cand_raw, "stg_raw_candidats")

df_cand_std   = df_cand_raw.copy()
df_cand_rejet = pd.DataFrame()
source_nom    = "stg_raw_candidats.csv"

# ── Standardisation ────────────────────────────────────────────────────────
# Code département : strip + zfill
df_cand_std['code_departement'] = (
    df_cand_std['code_departement']
    .astype(str).str.strip().str.zfill(2)
)

# id_election : strip + lower
df_cand_std['id_election'] = (
    df_cand_std['id_election']
    .astype(str).str.strip().str.lower()
)

# Nuance : strip + upper (pour harmoniser "RN", "rn", " RN ")
df_cand_std['nuance'] = (
    df_cand_std['nuance']
    .astype(str).str.strip().str.upper()
)

# Nom / prénom : strip + title case
for col in ['nom', 'prenom']:
    if col in df_cand_std.columns:
        df_cand_std[col] = df_cand_std[col].astype(str).str.strip().str.title()

# Colonnes numériques
for col in ['voix', 'ratio_voix_inscrits', 'ratio_voix_exprimes']:
    if col in df_cand_std.columns:
        df_cand_std[col] = pd.to_numeric(df_cand_std[col], errors='coerce')

print("✅ Standardisation candidats appliquée")


C:\Users\OMEN\AppData\Local\Temp\ipykernel_31604\3784866038.py:2: DtypeWarning: Columns (0: code_bv, 1: nuance, 2: sexe) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cand_raw = pd.read_csv(


📂 Chargé : stg_raw_candidats.csv — 222,280 lignes
  📊 Score qualité — stg_raw_candidats
     Complétude  : 68.64% (1,254,840 nulls sur 4,001,040 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 81.18%
  ⚠️  ALERTE : score qualité < 90% !

✅ Standardisation candidats appliquée


In [18]:
# ── Contrôles qualité candidats ───────────────────────────────────────────

# R01 : code_departement hors PDL
masque_r01 = ~df_cand_std['code_departement'].isin(DEPTS_PDL)
if masque_r01.any():
    df_cand_rejet = creer_rejet(df_cand_rejet, BATCH_ID, source_nom,
        'R01', 'code_departement hors PDL', df_cand_std[masque_r01])
    df_cand_std = df_cand_std[~masque_r01].copy()
    print(f"  R01 : {masque_r01.sum():,} rejetées (dept hors PDL)")

# R02 : id_election hors cibles
masque_r02 = ~df_cand_std['id_election'].isin(ELECTIONS_CIBLES)
if masque_r02.any():
    df_cand_rejet = creer_rejet(df_cand_rejet, BATCH_ID, source_nom,
        'R02', 'id_election hors élections cibles', df_cand_std[masque_r02])
    df_cand_std = df_cand_std[~masque_r02].copy()
    print(f"  R02 : {masque_r02.sum():,} rejetées (élection hors cibles)")

# R06 : nuance null (on ne peut pas classifier politiquement)
masque_r06 = df_cand_std['nuance'].isnull() | (df_cand_std['nuance'] == 'NAN')
if masque_r06.any():
    df_cand_rejet = creer_rejet(df_cand_rejet, BATCH_ID, source_nom,
        'R06', 'nuance politique null — classification impossible', df_cand_std[masque_r06])
    df_cand_std = df_cand_std[~masque_r06].copy()
    print(f"  R06 : {masque_r06.sum():,} rejetées (nuance null)")

# R07 : voix null ou < 0
masque_r07 = df_cand_std['voix'].isnull() | (df_cand_std['voix'] < 0)
if masque_r07.any():
    df_cand_rejet = creer_rejet(df_cand_rejet, BATCH_ID, source_nom,
        'R07', 'voix null ou négatif', df_cand_std[masque_r07])
    df_cand_std = df_cand_std[~masque_r07].copy()
    print(f"  R07 : {masque_r07.sum():,} rejetées (voix invalides)")

# R08 : ratio_voix_exprimes hors [0, 100]
masque_r08 = (
    df_cand_std['ratio_voix_exprimes'].isnull() |
    (df_cand_std['ratio_voix_exprimes'] < 0) |
    (df_cand_std['ratio_voix_exprimes'] > 100)
)
if masque_r08.any():
    df_cand_rejet = creer_rejet(df_cand_rejet, BATCH_ID, source_nom,
        'R08', 'ratio_voix_exprimes hors plage [0, 100]', df_cand_std[masque_r08])
    df_cand_std = df_cand_std[~masque_r08].copy()
    print(f"  R08 : {masque_r08.sum():,} rejetées (ratio hors plage)")

print()
rapport_std(df_cand_raw, df_cand_std, df_cand_rejet, source_nom)


  R06 : 77,050 rejetées (nuance null)

  📋 Rapport staging — stg_raw_candidats.csv
     Entrée (stg_raw)  : 222,280 lignes
     Valides (stg_std) : 145,230 lignes
     Rejetées          : 77,050 lignes
     Taux de rejet     : 34.66%
  ⚠️  ALERTE : taux de rejet > 10% !



In [19]:
# ── Enrichissement : famille politique ────────────────────────────────────
# On regroupe les nuances en grandes familles politiques
# pour faciliter l'analyse "qui gagne" dans le notebook 03

FAMILLES_POLITIQUES = {
    # Gauche
    'HOLL': 'GAUCHE', 'SOC': 'GAUCHE', 'DVG': 'GAUCHE', 'RDG': 'GAUCHE',
    'PS' : 'GAUCHE',
    # Gauche radicale
    'MELE': 'GAUCHE_RADICALE', 'FI': 'GAUCHE_RADICALE', 'EXG': 'GAUCHE_RADICALE',
    'COM': 'GAUCHE_RADICALE', 'LO': 'GAUCHE_RADICALE', 'NUP': 'GAUCHE_RADICALE',
    'FG' : 'GAUCHE_RADICALE', 'DXG': 'GAUCHE_RADICALE',
    # Centre
    'BAYR': 'CENTRE', 'MACR': 'CENTRE', 'REM': 'CENTRE', 'ENS': 'CENTRE',
    'MDM': 'CENTRE', 'UDI': 'CENTRE', 'CEN': 'CENTRE',
    # Droite
    'SARK': 'DROITE', 'UMP': 'DROITE', 'LR': 'DROITE', 'DVD': 'DROITE',
    'NCE': 'DROITE', 'MAJ': 'DROITE', 'RPF': 'DROITE',
    # Extrême droite
    'LEPE': 'EXTREME_DROITE', 'FN': 'EXTREME_DROITE', 'RN': 'EXTREME_DROITE',
    'EXD': 'EXTREME_DROITE', 'REC': 'EXTREME_DROITE', 'DLF': 'EXTREME_DROITE',
    # Écologie
    'ECO': 'ECOLOGIE', 'JOLY': 'ECOLOGIE', 'VEC': 'ECOLOGIE',
    # Divers
    'DIV': 'DIVERS', 'REG': 'DIVERS', 'AUT': 'DIVERS', 'DSV': 'DIVERS',
    'DVC': 'DIVERS', 'PRV': 'DIVERS',
}

df_cand_std['famille_politique'] = (
    df_cand_std['nuance']
    .map(FAMILLES_POLITIQUES)
    .fillna('AUTRE')
)

# Extraction année et type élection
df_cand_std['annee']         = df_cand_std['id_election'].str[:4].astype(int)
df_cand_std['type_election'] = df_cand_std['id_election'].str[5:9]

print("✅ Familles politiques assignées")
print()
print("  Répartition des familles :")
print(df_cand_std['famille_politique'].value_counts().to_string())


✅ Familles politiques assignées

  Répartition des familles :
famille_politique
GAUCHE_RADICALE    29793
DROITE             22044
EXTREME_DROITE     20477
DIVERS             16428
ECOLOGIE           15959
GAUCHE             13819
AUTRE              13814
CENTRE             12896


In [20]:
# ── Sauvegarde candidats ──────────────────────────────────────────────────
df_cand_std.to_csv(
    os.path.join(PATH_STG_STD, "stg_std_candidats.csv"),
    index=False, encoding='utf-8'
)
df_cand_rejet.to_csv(
    os.path.join(PATH_STG_REJECT, "stg_reject_candidats.csv"),
    index=False, encoding='utf-8'
)

print(f"💾 stg_std_candidats.csv    → {len(df_cand_std):,} lignes")
print(f"💾 stg_reject_candidats.csv → {len(df_cand_rejet):,} lignes rejetées")


💾 stg_std_candidats.csv    → 145,230 lignes
💾 stg_reject_candidats.csv → 77,050 lignes rejetées


---
## 4. Staging — `stg_raw_securite.csv`
### Indicateur criminalité

**Règles de rejet :**
- R01 : `code_departement` hors PDL
- R09 : `annee` null ou hors plage [2010, 2023]
- R10 : `taux_pour_mille` null (on ne peut pas calculer l'indicateur)


In [21]:
# ── Chargement et standardisation sécurité ────────────────────────────────
df_secu_raw = pd.read_csv(
    os.path.join(PATH_STG_RAW, "stg_raw_securite.csv"),
    dtype={'CODGEO_2025': str, 'code_departement': str},
    sep=','
)

print(f"📂 Chargé : stg_raw_securite.csv — {len(df_secu_raw):,} lignes")
score_qualite(df_secu_raw, "stg_raw_securite")

df_secu_std   = df_secu_raw.copy()
df_secu_rejet = pd.DataFrame()
source_nom    = "stg_raw_securite.csv"

# Standardisation
df_secu_std['code_departement'] = (
    df_secu_std['code_departement'].astype(str).str.strip().str.zfill(2)
)
df_secu_std['annee'] = pd.to_numeric(df_secu_std['annee'], errors='coerce')

# Nettoyer taux_pour_mille : remplacer virgule par point (format français)
df_secu_std['taux_pour_mille'] = (
    df_secu_std['taux_pour_mille']
    .astype(str).str.replace(',', '.', regex=False)
)
df_secu_std['taux_pour_mille'] = pd.to_numeric(
    df_secu_std['taux_pour_mille'], errors='coerce'
)

df_secu_std['indicateur'] = df_secu_std['indicateur'].astype(str).str.strip()

print("✅ Standardisation sécurité appliquée")


📂 Chargé : stg_raw_securite.csv — 73,680 lignes
  📊 Score qualité — stg_raw_securite
     Complétude  : 85.71% (147,360 nulls sur 1,031,520 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 91.43%
  ✅ Qualité acceptable

✅ Standardisation sécurité appliquée


In [22]:
# ── Contrôles qualité sécurité ────────────────────────────────────────────

# R01 : code_departement hors PDL
masque_r01 = ~df_secu_std['code_departement'].isin(DEPTS_PDL)
if masque_r01.any():
    df_secu_rejet = creer_rejet(df_secu_rejet, BATCH_ID, source_nom,
        'R01', 'code_departement hors PDL', df_secu_std[masque_r01])
    df_secu_std = df_secu_std[~masque_r01].copy()

# R09 : annee invalide
masque_r09 = df_secu_std['annee'].isnull() | (df_secu_std['annee'] < 2010)
if masque_r09.any():
    df_secu_rejet = creer_rejet(df_secu_rejet, BATCH_ID, source_nom,
        'R09', 'annee null ou hors plage [2010-2023]', df_secu_std[masque_r09])
    df_secu_std = df_secu_std[~masque_r09].copy()
    print(f"  R09 : {masque_r09.sum():,} rejetées (année invalide)")

# R10 : taux_pour_mille null
# On garde les lignes avec taux null mais on les met en quarantaine
# (elles peuvent être utiles pour les statistiques de complétude)
masque_r10 = df_secu_std['taux_pour_mille'].isnull()
if masque_r10.any():
    df_secu_rejet = creer_rejet(df_secu_rejet, BATCH_ID, source_nom,
        'R10', 'taux_pour_mille null — indicateur non calculable',
        df_secu_std[masque_r10])
    df_secu_std = df_secu_std[~masque_r10].copy()
    print(f"  R10 : {masque_r10.sum():,} rejetées (taux null)")

print()
rapport_std(df_secu_raw, df_secu_std, df_secu_rejet, source_nom)


  R10 : 39,599 rejetées (taux null)

  📋 Rapport staging — stg_raw_securite.csv
     Entrée (stg_raw)  : 73,680 lignes
     Valides (stg_std) : 34,081 lignes
     Rejetées          : 39,599 lignes
     Taux de rejet     : 53.74%
  ⚠️  ALERTE : taux de rejet > 10% !



In [23]:
# ── Sauvegarde sécurité ───────────────────────────────────────────────────
df_secu_std.to_csv(
    os.path.join(PATH_STG_STD, "stg_std_securite.csv"),
    index=False, encoding='utf-8'
)
df_secu_rejet.to_csv(
    os.path.join(PATH_STG_REJECT, "stg_reject_securite.csv"),
    index=False, encoding='utf-8'
)

print(f"💾 stg_std_securite.csv    → {len(df_secu_std):,} lignes")
print(f"💾 stg_reject_securite.csv → {len(df_secu_rejet):,} lignes rejetées")


💾 stg_std_securite.csv    → 34,081 lignes
💾 stg_reject_securite.csv → 39,599 lignes rejetées


---
## 5. Staging — `stg_raw_emploi.csv`
### Indicateur chômage

**Règles de rejet :**
- R01 : `code_departement` hors PDL
- R11 : `CODGEO` null (commune non identifiable)
- R12 : Tous les chômeurs nulls sur les 3 millésimes (commune sans données)


In [24]:
# ── Chargement et standardisation emploi ──────────────────────────────────
df_emp_raw = pd.read_csv(
    os.path.join(PATH_STG_RAW, "stg_raw_emploi.csv"),
    dtype={'CODGEO': str, 'code_departement': str},
    sep=','
)

print(f"📂 Chargé : stg_raw_emploi.csv — {len(df_emp_raw):,} lignes")
score_qualite(df_emp_raw, "stg_raw_emploi")

df_emp_std   = df_emp_raw.copy()
df_emp_rejet = pd.DataFrame()
source_nom   = "stg_raw_emploi.csv"

# Standardisation
df_emp_std['code_departement'] = (
    df_emp_std['code_departement'].astype(str).str.strip().str.zfill(2)
)
df_emp_std['CODGEO'] = df_emp_std['CODGEO'].astype(str).str.strip()

# Typer toutes les colonnes numériques
cols_num = [c for c in df_emp_std.columns if c not in ['CODGEO', 'code_departement']]
for col in cols_num:
    df_emp_std[col] = pd.to_numeric(df_emp_std[col], errors='coerce')

print("✅ Standardisation emploi appliquée")


📂 Chargé : stg_raw_emploi.csv — 1,232 lignes
  📊 Score qualité — stg_raw_emploi
     Complétude  : 99.88% (12 nulls sur 9,856 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 99.93%
  ✅ Qualité acceptable

✅ Standardisation emploi appliquée


In [25]:
# ── Contrôles qualité emploi ──────────────────────────────────────────────

# R01 : département hors PDL
masque_r01 = ~df_emp_std['code_departement'].isin(DEPTS_PDL)
if masque_r01.any():
    df_emp_rejet = creer_rejet(df_emp_rejet, BATCH_ID, source_nom,
        'R01', 'code_departement hors PDL', df_emp_std[masque_r01])
    df_emp_std = df_emp_std[~masque_r01].copy()

# R11 : CODGEO null
masque_r11 = df_emp_std['CODGEO'].isnull() | (df_emp_std['CODGEO'] == 'nan')
if masque_r11.any():
    df_emp_rejet = creer_rejet(df_emp_rejet, BATCH_ID, source_nom,
        'R11', 'CODGEO null — commune non identifiable', df_emp_std[masque_r11])
    df_emp_std = df_emp_std[~masque_r11].copy()
    print(f"  R11 : {masque_r11.sum():,} rejetées (CODGEO null)")

# R12 : toutes les colonnes chômage nulles
cols_chom = [c for c in df_emp_std.columns if 'CHOM' in c]
if cols_chom:
    masque_r12 = df_emp_std[cols_chom].isnull().all(axis=1)
    if masque_r12.any():
        df_emp_rejet = creer_rejet(df_emp_rejet, BATCH_ID, source_nom,
            'R12', 'Tous les indicateurs chômage sont null', df_emp_std[masque_r12])
        df_emp_std = df_emp_std[~masque_r12].copy()
        print(f"  R12 : {masque_r12.sum():,} rejetées (pas de données chômage)")

# ── Calcul du taux de chômage pour chaque millésime ──────────────────────
# Formule : taux = (chômeurs / population active) × 100
for millesime, annee_proxy in [('P10', 2012), ('P15', 2017), ('P21', 2022)]:
    col_chom = f'{millesime}_CHOM1564'
    col_pop  = f'{millesime}_POP1564'
    col_taux = f'taux_chomage_{annee_proxy}'
    
    if col_chom in df_emp_std.columns and col_pop in df_emp_std.columns:
        df_emp_std[col_taux] = (
            df_emp_std[col_chom] / df_emp_std[col_pop] * 100
        ).round(4)
        print(f"  ✅ taux_chomage_{annee_proxy} calculé")

print()
rapport_std(df_emp_raw, df_emp_std, df_emp_rejet, source_nom)


  R12 : 2 rejetées (pas de données chômage)
  ✅ taux_chomage_2012 calculé
  ✅ taux_chomage_2017 calculé
  ✅ taux_chomage_2022 calculé

  📋 Rapport staging — stg_raw_emploi.csv
     Entrée (stg_raw)  : 1,232 lignes
     Valides (stg_std) : 1,230 lignes
     Rejetées          : 2 lignes
     Taux de rejet     : 0.16%
  ✅ Taux de rejet acceptable



In [26]:
# ── Sauvegarde emploi ─────────────────────────────────────────────────────
df_emp_std.to_csv(
    os.path.join(PATH_STG_STD, "stg_std_emploi.csv"),
    index=False, encoding='utf-8'
)
df_emp_rejet.to_csv(
    os.path.join(PATH_STG_REJECT, "stg_reject_emploi.csv"),
    index=False, encoding='utf-8'
)

print(f"💾 stg_std_emploi.csv    → {len(df_emp_std):,} lignes")
print(f"💾 stg_reject_emploi.csv → {len(df_emp_rejet):,} lignes rejetées")


💾 stg_std_emploi.csv    → 1,230 lignes
💾 stg_reject_emploi.csv → 2 lignes rejetées


---
## 6. Staging — `stg_raw_socioeco.csv`
### Indicateurs socio-économiques (pauvreté, population, entreprises)

**Règles de rejet :**
- R01 : `code_departement` hors PDL
- R11 : `CODGEO` null
- R13 : `P22_POP` null ou <= 0 (commune sans population = inutilisable)


In [27]:
# ── Chargement et standardisation socio-éco ──────────────────────────────
df_soc_raw = pd.read_csv(
    os.path.join(PATH_STG_RAW, "stg_raw_socioeco.csv"),
    dtype={'CODGEO': str, 'code_departement': str},
    sep=','
)

print(f"📂 Chargé : stg_raw_socioeco.csv — {len(df_soc_raw):,} lignes")
score_qualite(df_soc_raw, "stg_raw_socioeco")

df_soc_std   = df_soc_raw.copy()
df_soc_rejet = pd.DataFrame()
source_nom   = "stg_raw_socioeco.csv"

# Standardisation
df_soc_std['code_departement'] = (
    df_soc_std['code_departement'].astype(str).str.strip().str.zfill(2)
)
df_soc_std['CODGEO'] = df_soc_std['CODGEO'].astype(str).str.strip()

# Nettoyer PR_MD60_23 : peut contenir 's' (secret statistique INSEE)
# On remplace 's' par NaN (non disponible, pas une erreur)
if 'PR_MD60_23' in df_soc_std.columns:
    df_soc_std['PR_MD60_23'] = df_soc_std['PR_MD60_23'].replace('s', np.nan)

# Typer les colonnes numériques
cols_num = [c for c in df_soc_std.columns if c not in ['CODGEO', 'code_departement']]
for col in cols_num:
    df_soc_std[col] = pd.to_numeric(df_soc_std[col], errors='coerce')

print("✅ Standardisation socio-éco appliquée")
print("   Note : valeurs 's' (secret statistique INSEE) → NaN")


📂 Chargé : stg_raw_socioeco.csv — 1,233 lignes
  📊 Score qualité — stg_raw_socioeco
     Complétude  : 99.67% (45 nulls sur 13,563 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 99.80%
  ✅ Qualité acceptable

✅ Standardisation socio-éco appliquée
   Note : valeurs 's' (secret statistique INSEE) → NaN


In [28]:
# ── Contrôles qualité socio-éco ───────────────────────────────────────────

# R01 : département hors PDL
masque_r01 = ~df_soc_std['code_departement'].isin(DEPTS_PDL)
if masque_r01.any():
    df_soc_rejet = creer_rejet(df_soc_rejet, BATCH_ID, source_nom,
        'R01', 'code_departement hors PDL', df_soc_std[masque_r01])
    df_soc_std = df_soc_std[~masque_r01].copy()

# R11 : CODGEO null
masque_r11 = df_soc_std['CODGEO'].isnull() | (df_soc_std['CODGEO'] == 'nan')
if masque_r11.any():
    df_soc_rejet = creer_rejet(df_soc_rejet, BATCH_ID, source_nom,
        'R11', 'CODGEO null — commune non identifiable', df_soc_std[masque_r11])
    df_soc_std = df_soc_std[~masque_r11].copy()

# R13 : population nulle ou négative
if 'P22_POP' in df_soc_std.columns:
    masque_r13 = df_soc_std['P22_POP'].isnull() | (df_soc_std['P22_POP'] <= 0)
    if masque_r13.any():
        df_soc_rejet = creer_rejet(df_soc_rejet, BATCH_ID, source_nom,
            'R13', 'P22_POP null ou <= 0 — commune inutilisable',
            df_soc_std[masque_r13])
        df_soc_std = df_soc_std[~masque_r13].copy()
        print(f"  R13 : {masque_r13.sum():,} rejetées (population nulle)")

print()
rapport_std(df_soc_raw, df_soc_std, df_soc_rejet, source_nom)


  R13 : 5 rejetées (population nulle)

  📋 Rapport staging — stg_raw_socioeco.csv
     Entrée (stg_raw)  : 1,233 lignes
     Valides (stg_std) : 1,228 lignes
     Rejetées          : 5 lignes
     Taux de rejet     : 0.41%
  ✅ Taux de rejet acceptable



In [29]:
# ── Sauvegarde socio-éco ──────────────────────────────────────────────────
df_soc_std.to_csv(
    os.path.join(PATH_STG_STD, "stg_std_socioeco.csv"),
    index=False, encoding='utf-8'
)
df_soc_rejet.to_csv(
    os.path.join(PATH_STG_REJECT, "stg_reject_socioeco.csv"),
    index=False, encoding='utf-8'
)

print(f"💾 stg_std_socioeco.csv    → {len(df_soc_std):,} lignes")
print(f"💾 stg_reject_socioeco.csv → {len(df_soc_rejet):,} lignes rejetées")


💾 stg_std_socioeco.csv    → 1,228 lignes
💾 stg_reject_socioeco.csv → 5 lignes rejetées


---
## 7. Bilan global du staging

In [30]:
# ── Bilan global de tous les rejets ──────────────────────────────────────
print("=" * 65)
print("  BILAN GLOBAL STAGING — stg_std + stg_reject")
print("=" * 65)
print()

datasets = {
    "general"  : (df_gen_raw,  df_gen_std,  df_gen_rejet),
    "candidats": (df_cand_raw, df_cand_std, df_cand_rejet),
    "securite" : (df_secu_raw, df_secu_std, df_secu_rejet),
    "emploi"   : (df_emp_raw,  df_emp_std,  df_emp_rejet),
    "socioeco" : (df_soc_raw,  df_soc_std,  df_soc_rejet),
}

bilan = []
scores = {}
for nom, (raw, std, rej) in datasets.items():
    n_entree = len(raw)
    n_valid  = len(std)
    n_rejet  = len(rej)
    taux_rej = n_rejet / n_entree * 100 if n_entree > 0 else 0
    score    = score_qualite(std, f"stg_std_{nom}")
    scores[nom] = score
    bilan.append({
        'dataset'     : nom,
        'entree'      : n_entree,
        'valides'     : n_valid,
        'rejetes'     : n_rejet,
        'taux_rejet%' : round(taux_rej, 2),
        'score_qualite': score
    })

df_bilan = pd.DataFrame(bilan)
print("\n  Tableau récapitulatif :")
print(df_bilan.to_string(index=False))
print()

# Alerte globale
datasets_alertes = df_bilan[df_bilan['taux_rejet%'] > SEUIL_REJET_MAX * 100]
if len(datasets_alertes) > 0:
    print(f"  ⚠️  ALERTES QUALITÉ :")
    for _, row in datasets_alertes.iterrows():
        print(f"     {row['dataset']} : taux de rejet {row['taux_rejet%']}% > seuil {SEUIL_REJET_MAX*100:.0f}%")
else:
    print("  ✅ Tous les taux de rejet sont dans les limites acceptables")


  BILAN GLOBAL STAGING — stg_std + stg_reject

  📊 Score qualité — stg_std_general
     Complétude  : 90.82% (8,591 nulls sur 93,612 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 94.49%
  ✅ Qualité acceptable

  📊 Score qualité — stg_std_candidats
     Complétude  : 74.01% (792,516 nulls sur 3,049,830 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 84.41%
  ⚠️  ALERTE : score qualité < 90% !

  📊 Score qualité — stg_std_securite
     Complétude  : 85.71% (68,162 nulls sur 477,134 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 91.43%
  ✅ Qualité acceptable

  📊 Score qualité — stg_std_emploi
     Complétude  : 100.00% (0 nulls sur 13,530 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 100.00%
  ✅ Qualité acceptable

  📊 Score qualité — stg_std_socioeco
     Complétude  : 91.60% (1,134 nulls sur 13,508 cellules)
     Unicité     : 100.00% (0 doublons)
     Score global: 94.96%
  ✅ Qualité acceptable




In [31]:
# ── Consolidation de tous les rejets dans un fichier unique ──────────────
# Un seul fichier de rejets global pour faciliter l'audit

tous_les_rejets = pd.concat([
    df_gen_rejet, df_cand_rejet, df_secu_rejet,
    df_emp_rejet, df_soc_rejet
], ignore_index=True)

chemin_rejet_global = os.path.join(PATH_STG_REJECT, "stg_reject_ALL.csv")
tous_les_rejets.to_csv(chemin_rejet_global, index=False, encoding='utf-8')

print(f"💾 Fichier de rejets consolidé : stg_reject_ALL.csv")
print(f"   Total rejets : {len(tous_les_rejets):,} lignes")
if len(tous_les_rejets) > 0:
    print()
    print("  Répartition des rejets par règle :")
    print(tous_les_rejets.groupby(['source_fichier','reject_rule','reject_reason'])
          ['nb_lignes'].sum().reset_index().to_string(index=False))


💾 Fichier de rejets consolidé : stg_reject_ALL.csv
   Total rejets : 133,252 lignes

  Répartition des rejets par règle :
       source_fichier reject_rule                                                         reject_reason  nb_lignes
stg_raw_candidats.csv         R06                     nuance politique null — classification impossible 5936702500
   stg_raw_emploi.csv         R12                                Tous les indicateurs chômage sont null          4
  stg_raw_general.csv         R05 Doublon sur clé métier ['id_election', 'code_departement', 'code_bv']  275427216
 stg_raw_securite.csv         R10                      taux_pour_mille null — indicateur non calculable 1568080801
 stg_raw_socioeco.csv         R13                           P22_POP null ou <= 0 — commune inutilisable         25


In [32]:
# ── Enregistrement batch control ─────────────────────────────────────────
batch_file = os.path.join(PATH_OPS, "ops_batch_control.csv")

batch_row = pd.DataFrame([{
    "batch_id"        : BATCH_ID,
    "notebook"        : "02_staging_qualite",
    "statut"          : "SUCCESS",
    "zone_etude"      : ZONE_ETUDE,
    "nb_fichiers_std" : len(datasets),
    "nb_rejets_total" : len(tous_les_rejets),
    "duree_sec"       : (datetime.now() - BATCH_START).seconds,
    "run_ts"          : datetime.now().isoformat(),
    "note"            : f"Staging OK — {len(tous_les_rejets)} rejets isolés dans stg_reject_ALL.csv"
}])

if os.path.exists(batch_file):
    df_existing = pd.read_csv(batch_file)
    batch_row   = pd.concat([df_existing, batch_row], ignore_index=True)

batch_row.to_csv(batch_file, index=False)
print(f"✅ Batch {BATCH_ID} enregistré")
print(f"   Durée : {(datetime.now() - BATCH_START).seconds} secondes")


✅ Batch B02_STAGING_20260523_213217 enregistré
   Durée : 61 secondes


---
## ✅ Récapitulatif — Ce qu'on a produit

### Fichiers `stg_std` (données nettoyées — pour les notebooks suivants)
| Fichier | Contenu |
|---|---|
| `stg_std_general.csv` | Participation + taux calculés + annee + type_election |
| `stg_std_candidats.csv` | Candidats + famille_politique + annee + type_election |
| `stg_std_securite.csv` | Criminalité nettoyée, taux en float |
| `stg_std_emploi.csv` | Chômage + taux_chomage_2012/2017/2022 calculés |
| `stg_std_socioeco.csv` | Socio-éco nettoyée, 's' INSEE → NaN |

### Fichiers `stg_reject` (traçabilité des anomalies)
| Fichier | Contenu |
|---|---|
| `stg_reject_ALL.csv` | Tous les rejets consolidés avec motif et payload |

---
> **Suite : Notebook 03 — Transformations métier**
> Agrégation par département, calcul des deltas, corrélations → `outputs/transformations/`
